# Practicum 1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from power_grid_model import (
    AttributeType,
    CalculationMethod,
    CalculationType,
    ComponentType,
    DatasetType,
    LoadGenType,
    PowerGridModel,
    initialize_array,
)
from power_grid_model.validation import assert_valid_input_data

## B1

In [ ]:
# node
node = initialize_array(DatasetType.input, ComponentType.node, 3)
node[AttributeType.id] = np.array([1, 2, 6])
node[AttributeType.u_rated] = [10.5e3, 10.5e3, 10.5e3]

# line
line = initialize_array(DatasetType.input, ComponentType.line, 3)
line[AttributeType.id] = [3, 5, 8]
line[AttributeType.from_node] = [1, 2, 1]
line[AttributeType.to_node] = [2, 6, 6]
line[AttributeType.from_status] = [1, 1, 1]
line[AttributeType.to_status] = [1, 1, 1]
line[AttributeType.r1] = [0.25, 0.25, 0.25]
line[AttributeType.x1] = [0.2, 0.2, 0.2]
line[AttributeType.c1] = [10e-6, 10e-6, 10e-6]
line[AttributeType.tan1] = [0.0, 0.0, 0.0]
line[AttributeType.i_n] = [1000, 1000, 1000]

# load
sym_load = initialize_array(DatasetType.input, ComponentType.sym_load, 2)
sym_load[AttributeType.id] = [4, 7]
sym_load[AttributeType.node] = [2, 6]
sym_load[AttributeType.status] = [1, 1]
sym_load[AttributeType.type] = [LoadGenType.const_power, LoadGenType.const_power]
sym_load[AttributeType.p_specified] = [20e6, 10e6]
sym_load[AttributeType.q_specified] = [5e6, 2e6]

# source
source = initialize_array(DatasetType.input, ComponentType.source, 1)
source[AttributeType.id] = [10]
source[AttributeType.node] = [1]
source[AttributeType.status] = [1]
source[AttributeType.u_ref] = [1.0]

# all
input_data = {
    ComponentType.node: node,
    ComponentType.line: line,
    ComponentType.sym_load: sym_load,
    ComponentType.source: source,
}

for component in input_data:
    print(f"------{component}------")
    print(pd.DataFrame(input_data[component]).dropna(axis=1, how="all"))

## B2

In [ ]:
assert_valid_input_data(input_data=input_data, calculation_type=CalculationType.power_flow)
print("Input data is valid")

## B3

In [ ]:
model = PowerGridModel(input_data)

## B4

In [ ]:
output_data = model.calculate_power_flow(
    symmetric=True, error_tolerance=1e-8, max_iterations=20, calculation_method=CalculationMethod.newton_raphson
)

## B5

In [ ]:
print("------node result------")
print(pd.DataFrame(output_data[ComponentType.node]))
print("------line result------")
print(pd.DataFrame(output_data[ComponentType.line]))

## B6

In [ ]:
node_result = pd.DataFrame(output_data[ComponentType.node])
line_result = pd.DataFrame(output_data[ComponentType.line])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(node_result["id"].astype(str), node_result["u_pu"], "o", markersize=8)
ax1.set_xlabel("Node")
ax1.set_ylabel("Voltage [pu]")
ax1.set_title("Node voltages")
ax2.bar(line_result["id"].astype(str), 100 * line_result["loading"], width=0.5)
ax2.axhline(100, color="grey", linestyle="--")
ax2.set_xlabel("Line")
ax2.set_ylabel("Loading [%]")
ax2.set_title("Line loading")
plt.tight_layout()
plt.show()

## B7

In [ ]:
output_data_linear = model.calculate_power_flow(symmetric=True, calculation_method=CalculationMethod.linear)

print("------node result------")
print(pd.DataFrame({
    "id": output_data[ComponentType.node][AttributeType.id],
    "u_pu NR": output_data[ComponentType.node][AttributeType.u_pu],
    "u_pu linear": output_data_linear[ComponentType.node][AttributeType.u_pu],
}))
print("------line result------")
print(pd.DataFrame({
    "id": output_data[ComponentType.line][AttributeType.id],
    "loading NR": output_data[ComponentType.line][AttributeType.loading],
    "loading linear": output_data_linear[ComponentType.line][AttributeType.loading],
}))
print("------sym_load result------")
print(pd.DataFrame({
    "id": output_data[ComponentType.sym_load][AttributeType.id],
    "p NR": output_data[ComponentType.sym_load][AttributeType.p],
    "p linear": output_data_linear[ComponentType.sym_load][AttributeType.p],
}))

The linear method treats loads as constant impedance, so it overestimates voltages and underestimates line loading (90.1 % vs 98.6 % on line 3).

## C

In [ ]:
topology = pd.read_csv("Practicum_1_Python/grid_topology.csv")
consumption = pd.read_csv("Practicum_1_Python/consumption_one_year_15min.csv", index_col="date")
pv = pd.read_csv("Practicum_1_Python/pv_one_year_15min.csv", index_col="date")
ev = pd.read_csv("Practicum_1_Python/ev_one_year_15min.csv", index_col="date")

# bus number of each customer: "Customer_12 (kW)" -> 12
customers = consumption.columns.str.extract(r"Customer_(\d+)", expand=False).astype(int).to_numpy()

# node (one per customer, same order as the profile columns)
node = initialize_array(DatasetType.input, ComponentType.node, len(customers))
node[AttributeType.id] = customers
node[AttributeType.u_rated] = 400.0

# line
line = initialize_array(DatasetType.input, ComponentType.line, len(topology))
line[AttributeType.id] = 1000 + np.arange(len(topology))
line[AttributeType.from_node] = topology["FROM"]
line[AttributeType.to_node] = topology["TO"]
line[AttributeType.from_status] = 1
line[AttributeType.to_status] = 1
line[AttributeType.r1] = topology["Raa (Ω)"]
line[AttributeType.x1] = topology["Xaa (Ω)"]
line[AttributeType.c1] = 0.0
line[AttributeType.tan1] = 0.0
line[AttributeType.i_n] = topology["Imax"]

# load (one per customer, P and Q are set per time step)
sym_load = initialize_array(DatasetType.input, ComponentType.sym_load, len(customers))
sym_load[AttributeType.id] = 2000 + customers
sym_load[AttributeType.node] = customers
sym_load[AttributeType.status] = 1
sym_load[AttributeType.type] = LoadGenType.const_power
sym_load[AttributeType.p_specified] = 0.0
sym_load[AttributeType.q_specified] = 0.0

# source (MV/LV transformer at bus 1)
source = initialize_array(DatasetType.input, ComponentType.source, 1)
source[AttributeType.id] = [3000]
source[AttributeType.node] = [1]
source[AttributeType.status] = [1]
source[AttributeType.u_ref] = [1.0]

# all
input_data = {
    ComponentType.node: node,
    ComponentType.line: line,
    ComponentType.sym_load: sym_load,
    ComponentType.source: source,
}

assert_valid_input_data(input_data=input_data, calculation_type=CalculationType.power_flow)

In [ ]:
def reactive_power(p, power_factor=0.95):
    # leading power factor: Q has the opposite sign of P
    return -p * np.tan(np.arccos(power_factor))


def run_time_series(profile_kw, n_steps=None):
    p = profile_kw.to_numpy()[:n_steps] * 1000  # kW -> W
    q = reactive_power(p)
    names = profile_kw.columns.str.replace(" (kW)", "")

    model = PowerGridModel(input_data)
    update_sym_load = initialize_array(DatasetType.update, ComponentType.sym_load, len(customers))
    update_sym_load[AttributeType.id] = 2000 + customers

    u_pu = np.zeros((len(p), len(customers)))
    loading = np.zeros((len(p), len(topology)))
    for t in range(len(p)):
        update_sym_load[AttributeType.p_specified] = p[t]
        update_sym_load[AttributeType.q_specified] = q[t]
        model.update(update_data={ComponentType.sym_load: update_sym_load})

        output_data = model.calculate_power_flow(
            symmetric=True, error_tolerance=1e-8, max_iterations=20, calculation_method=CalculationMethod.newton_raphson
        )
        u_pu[t] = output_data[ComponentType.node][AttributeType.u_pu]
        loading[t] = output_data[ComponentType.line][AttributeType.loading]
        voltages = {"time step": profile_kw.index[t], "consumers": dict(zip(names, u_pu[t].tolist()))}

    index = profile_kw.index[: len(p)]
    u_pu = pd.DataFrame(u_pu, index=index, columns=names)
    loading = pd.DataFrame(loading, index=index, columns=topology["FROM"].astype(str) + "-" + topology["TO"].astype(str))
    return u_pu, loading, voltages


def summary(u_pu, loading):
    return pd.Series({
        "min voltage [pu]": u_pu.min().min(),
        "max voltage [pu]": u_pu.max().max(),
        "max line loading [%]": 100 * loading.max().max(),
        "lines overloaded at least once": (loading.max() > 1).sum(),
        "hours with an overloaded line": 0.25 * (loading.max(axis=1) > 1).sum(),
    })

In [ ]:
u_test, loading_test, voltages = run_time_series(consumption, n_steps=4)
print(voltages)

In [ ]:
u_base, loading_base, voltages = run_time_series(consumption)
print(voltages)
print(summary(u_base, loading_base))

## D

In [ ]:
combined = consumption + ev - pv
combined.to_csv("Practicum_1_Python/net_load_pv_ev_one_year_15min.csv", float_format="%.6f")

u_pv_ev, loading_pv_ev, voltages = run_time_series(combined)
print(summary(u_pv_ev, loading_pv_ev))

In [ ]:
# voltage magnitudes with and without PV/EV
total = combined.sum(axis=1)
shown = ["Customer_1", "Customer_20", "Customer_66", "Customer_91", "Customer_95"]
for t in [total.idxmin(), total.idxmax()]:  # largest PV export, largest demand
    print(f"------{t}, total net load {total[t]:.0f} kW------")
    print(pd.DataFrame({"without PV/EV": u_base.loc[t, shown], "with PV/EV": u_pv_ev.loc[t, shown]}))

In [ ]:
# line loadings with and without PV/EV (10 most loaded lines)
print(pd.DataFrame({
    "max loading without PV/EV": loading_base.max(),
    "max loading with PV/EV": loading_pv_ev.max(),
}).sort_values("max loading with PV/EV", ascending=False).head(10))
print("Highest loading at:", loading_pv_ev.max(axis=1).idxmax())

In [ ]:
# flexible setup: 1 = customer owns PV / EV charger, 0 = not
has_pv = pd.Series(1, index=consumption.columns)
has_ev = pd.Series(1, index=consumption.columns)
has_ev[["Customer_10 (kW)", "Customer_20 (kW)"]] = 0  # PV but no EV charger at buses 10 and 20

flexible = consumption + ev * has_ev - pv * has_pv
u_flex, loading_flex, _ = run_time_series(flexible)
print(summary(u_flex, loading_flex))

## Discussion

**Impact on voltage.** Without PV/EV, the voltages stay between 0.994 and 1.000 pu. With uncontrolled PV/EV they vary between 0.985 and 1.078 pu, which is still inside the 0.90–1.10 pu limits.

**Impact on line loading.** The highest line loading goes from 27 % without PV/EV to 372 % with PV/EV, on line 2-1 at the transformer. 33 lines are overloaded at some point, so the main problem is congestion.

**Same or different effects? Different.** PV raises the voltage at midday because power flows back to the transformer. EV charging lowers it in the evening. Both increase line loading, but the overloads come from PV: the highest loading occurs at the moment of largest PV export, because all PV systems produce at the same time.

**Flexible setup.** Give each customer an ownership flag `has_pv` / `has_ev` (1 or 0) and use net load = consumption + has_ev · EV − has_pv · PV. For the example, set `has_ev = 0` for the customers at buses 10 and 20.